In [1]:
#Imports & Φόρτωση Κανονικοποιημένων Δεδομένων
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform

os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

X_scaled_df = pd.read_csv('../data/processed/data_scaled.csv')
X_scaled = X_scaled_df.values
print(f"Διαστάσεις κανονικοποιημένων δεδομένων: {X_scaled.shape}")

Διαστάσεις κανονικοποιημένων δεδομένων: (400, 5)


In [2]:
# Συνάρτηση Υπολογισμού C-Index (Hubert & Levin, 1976)
def compute_c_index(X, labels):
    dists = squareform(pdist(X, metric='euclidean'))
    iu = np.triu_indices(len(X), k=1)
    all_dists = dists[iu]
    
    same_cluster = (labels[:, None] == labels[None, :])[iu]
    N_w = int(np.sum(same_cluster))
    if N_w == 0 or N_w == len(all_dists):
        return np.nan
        
    S_w = np.sum(all_dists[same_cluster])
    sorted_dists = np.sort(all_dists)
    S_min = np.sum(sorted_dists[:N_w])
    S_max = np.sum(sorted_dists[-N_w:])
    
    return (S_w - S_min) / (S_max - S_min)

In [ ]:
# Εκτέλεση K-Means (n_init=500) & Ward-D2 για k = 2..12
k_values = range(2, 13)
metrics_records = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, n_init=500, max_iter=500, random_state=42)
    labels_km = kmeans.fit_predict(X_scaled)
    
    ward = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage='ward')
    labels_ward = ward.fit_predict(X_scaled)
    
    wcss_ward = sum(
        np.sum((X_scaled[labels_ward == c] - X_scaled[labels_ward == c].mean(axis=0)) ** 2)
        for c in range(k)
    )
    
    metrics_records.append({
        'k': k,
        'WCSS_KMeans': kmeans.inertia_,
        'WCSS_Ward': wcss_ward,
        'C_Index_KMeans': compute_c_index(X_scaled, labels_km),
        'C_Index_Ward': compute_c_index(X_scaled, labels_ward),
        'Silhouette_KMeans': silhouette_score(X_scaled, labels_km),
        'Silhouette_Ward': silhouette_score(X_scaled, labels_ward),
        'CH_KMeans': calinski_harabasz_score(X_scaled, labels_km),
        'CH_Ward': calinski_harabasz_score(X_scaled, labels_ward),
        'DB_KMeans': davies_bouldin_score(X_scaled, labels_km),
        'DB_Ward': davies_bouldin_score(X_scaled, labels_ward)
    })

metrics_df = pd.DataFrame(metrics_records)
display(metrics_df.round(4))

,k,WCSS_KMeans,WCSS_Ward,C_Index_KMeans,C_Index_Ward,Silhouette_KMeans,Silhouette_Ward,CH_KMeans,CH_Ward,DB_KMeans,DB_Ward
0,2,1272.7048,1279.5223,0.1325,0.1136,0.4372,0.4595,227.4396,224.1072,0.7892,0.6746
1,3,834.6492,844.8266,0.1157,0.1163,0.3910,0.3933,277.1489,271.4189,1.0291,1.0362
2,4,598.4224,634.0266,0.0805,0.0897,0.4198,0.4468,309.1600,284.3863,1.0000,0.8276
3,5,456.5286,466.7714,0.0621,0.0614,0.4660,0.4599,333.8625,324.3694,0.8348,0.8532
4,6,369.6161,381.6344,0.0529,0.0534,0.4700,0.4627,347.5884,334.1607,0.8759,0.8967
5,7,308.6468,320.8523,0.0405,0.0438,0.4893,0.4824,358.9334,342.7875,0.8218,0.8162
6,8,259.7694,268.2357,0.0345,0.0413,0.5246,0.4969,375.1516,361.5432,0.7287,0.7817
7,9,210.9813,219.9189,0.0311,0.0318,0.5423,0.5338,414.4362,395.6071,0.6908,0.6998
8,10,180.1860,189.4482,0.0282,0.0303,0.5320,0.5320,437.6510,414.1355,0.6825,0.6762
9,11,152.0201,159.8429,0.0237,0.0261,0.5420,0.5255,472.8743,447.8279,0.6600,0.6673


In [4]:
# Αποθήκευση Ετικετών & PCA 2D με Ελλείψεις Εμπιστοσύνης 95%
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var_ratio = pca.explained_variance_ratio_ * 100

def plot_confidence_ellipse(x, y, ax, n_std=2.4477, facecolor='none', **kwargs):
    if x.size < 3:
        return None
    cov = np.cov(x, y)
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1] + 1e-12)
    ell_radius_x = np.sqrt(max(0.0, 1 + pearson))
    ell_radius_y = np.sqrt(max(0.0, 1 - pearson))
    
    ellipse = Ellipse((0, 0), width=ell_radius_x * 2, height=ell_radius_y * 2, facecolor=facecolor, **kwargs)
    scale_x = np.sqrt(cov[0, 0]) * n_std
    scale_y = np.sqrt(cov[1, 1]) * n_std
    transf = transforms.Affine2D().rotate_deg(45).scale(scale_x, scale_y).translate(np.mean(x), np.mean(y))
    ellipse.set_transform(transf + ax.transData)
    return ax.add_patch(ellipse)

cluster_results = pd.DataFrame(index=X_scaled_df.index)
for k in [3, 5, 9]:
    cluster_results[f'KMeans_k{k}'] = KMeans(n_clusters=k, n_init=500, random_state=42).fit_predict(X_scaled)
    cluster_results[f'Ward_k{k}'] = AgglomerativeClustering(n_clusters=k, metric='euclidean', linkage='ward').fit_predict(X_scaled)

cluster_results.to_csv('../data/processed/ml_cluster_labels.csv', index=False)